In [ ]:
#!/usr/bin/env python
# coding: utf-8

import cv2
import numpy as np
import time
import os
from datetime import datetime
import requests
from pythonping import ping
from selenium import webdriver
from ultralytics import YOLO
import glob
from scipy.spatial import distance as dist
from collections import OrderedDict
from typing import Optional
import logging
import csv
import threading

CCTV_URL = "1a0.18.28.119"
Camera_name = "CCTV DEPAN"
CSV_FILENAME = "polygon_coordinates.csv"
DEFAULT_TIMEOUT = 10

class CentroidTracker:
    def __init__(self, maxDisappeared=50):
        self.nextObjectID = 0
        self.objects = OrderedDict()
        self.disappeared = OrderedDict()
        self.maxDisappeared = maxDisappeared

    def register(self, centroid):
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.nextObjectID += 1

    def deregister(self, objectID):
        del self.objects[objectID]
        del self.disappeared[objectID]

    def update(self, inputCentroids):
        if len(inputCentroids) == 0:
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1
                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)
            return self.objects

        if len(self.objects) == 0:
            for i in range(0, len(inputCentroids)):
                self.register(inputCentroids[i])
        else:
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())

            D = dist.cdist(np.array(objectCentroids), inputCentroids)
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            usedRows = set()
            usedCols = set()

            for (row, col) in zip(rows, cols):
                if row in usedRows or col in usedCols:
                    continue

                objectID = objectIDs[row]
                self.objects[objectID] = inputCentroids[col]
                self.disappeared[objectID] = 0

                usedRows.add(row)
                usedCols.add(col)

            unusedRows = set(range(0, D.shape[0])).difference(usedRows)
            unusedCols = set(range(0, D.shape[1])).difference(usedCols)

            if D.shape[0] >= D.shape[1]:
                for row in unusedRows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1

                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unusedCols:
                    self.register(inputCentroids[col])

        return self.objects

def save_coordinates_to_csv(polygons):
    """Save polygon coordinates to CSV file"""
    with open(CSV_FILENAME, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['polygon_id', 'point_id', 'x', 'y'])
        for poly_id, polygon in enumerate(polygons):
            for point_id, (x, y) in enumerate(polygon):
                writer.writerow([poly_id, point_id, x, y])

def load_coordinates_from_csv():
    """Load polygon coordinates from CSV file"""
    polygons = []
    current_polygon = []
    current_poly_id = -1

    try:
        with open(CSV_FILENAME, 'r') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                poly_id = int(row['polygon_id'])
                x = int(row['x'])
                y = int(row['y'])

                if poly_id != current_poly_id:
                    if current_polygon:
                        polygons.append(current_polygon)
                    current_polygon = []
                    current_poly_id = poly_id

                current_polygon.append((x, y))

            if current_polygon:
                polygons.append(current_polygon)

        return polygons
    except FileNotFoundError:
        print(f"No existing {CSV_FILENAME} found.")
        return []

def create_detection_areas(polygons, image_shape):
    """Create a mask for multiple detection areas based on clicked points"""
    area_mask = np.zeros(image_shape[:2], dtype=np.uint8)
    for polygon in polygons:
        if len(polygon) == 4:  # Ensure polygon has exactly 4 points
            pts = np.array(polygon, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.fillPoly(area_mask, [pts], 255)
    return area_mask

def check_point_in_area(centroid, area_mask):
    """Check if a point falls within any detection area"""
    x, y = int(centroid[0]), int(centroid[1])
    if 0 <= x < area_mask.shape[1] and 0 <= y < area_mask.shape[0]:
        return area_mask[y, x] > 0
    return False

def get_points(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        image = param['image']
        current_polygon = param['current_polygon']
        polygons = param['polygons']

        # Add point to current polygon
        current_polygon.append((x, y))

        # Draw the point
        cv2.circle(image, (x, y), 5, (0, 0, 255), -1)

        # If we have more than 1 point in current polygon, draw line to previous point
        if len(current_polygon) > 1:
            cv2.line(image, current_polygon[-2], current_polygon[-1], (0, 255, 0), 2)

        # If we have 4 points, complete the polygon
        if len(current_polygon) == 4:
            # Draw final line to close the polygon
            cv2.line(image, current_polygon[0], current_polygon[-1], (0, 255, 0), 2)
            # Add completed polygon to list of polygons
            polygons.append(current_polygon.copy())
            # Save coordinates to CSV after each polygon completion
            save_coordinates_to_csv(polygons)
            # Clear current polygon for next one
            current_polygon.clear()
            print(f"Polygon {len(polygons)} completed and saved to CSV. Start new polygon or press 'q' to finish.")

        cv2.imshow("Screenshot", image)

def cleanup_directory(directory, pattern, keep_last_n=5):
    files = glob.glob(os.path.join(directory, pattern))
    files.sort(key=os.path.getctime)

    files_to_remove = files[:-keep_last_n] if len(files) > keep_last_n else []
    for file in files_to_remove:
        try:
            os.remove(file)
        except Exception as e:
            print(f"Error removing file {file}: {e}")

def check_cctv_connection():
    try:
        ip = CCTV_URL
        ping_result = ping(ip, count=1, verbose=True)
        if ping_result.success():
            print("CCTV is still connected")
            return True
        else:
            static_directory = os.path.join(os.path.dirname(os.path.dirname(__file__)),"standalone",  "image_dc.png")
            create_alarm_api(description=f"CCTV Disconnected from AI: {CCTV_URL}", image_path=static_directory)
            print("CCTV is disconnected")
            return False
    except Exception as e:
        print(f"Error checking CCTV connection: {e}")
        return False

def create_alarm_api(description: str, image_path: Optional[str] = None) -> dict:
    params = {
        "camera_name": Camera_name,
        "ip_address": CCTV_URL,
        "description": description,
    }

    files = {"image": open(image_path, "rb")} if image_path else None

    try:
        response = requests.post('http://localhost:8000/alarms/', params=params, files=files)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        logging.error("Error in create_alarm_api for CCTV_URL %s: %s", CCTV_URL, str(e))
        return {"error": str(e)}
    finally:
        if files and files["image"]:
            files["image"].close()

class DetectionMetrics:
    def __init__(self):
        self.reset()
        self.last_api_call = time.time()
        self.crossed_since_last_call = 0

    def reset(self):
        self.total_detected = 0
        self.crossed_count = 0
        self.not_crossed_count = 0

    def update(self, crossed):
        self.total_detected += 1
        if crossed:
            self.crossed_count += 1
            self.crossed_since_last_call += 1
        else:
            self.not_crossed_count += 1

def get_user_choice(timeout):
    """Get user choice with timeout"""
    choice = [None]
    timer = [timeout]

    def input_thread():
        while timer[0] > 0 and choice[0] is None:
            if choice[0] is None:
                print(f"\rChoose method (1: Manual click, 2: Load from CSV) - {timer[0]} seconds remaining", end='')
                if cv2.waitKey(1000) & 0xFF in [ord('1'), ord('2')]:
                    choice[0] = '1' if cv2.waitKey(0) & 0xFF == ord('1') else '2'
            timer[0] -= 1

    thread = threading.Thread(target=input_thread)
    thread.daemon = True
    thread.start()
    thread.join(timeout)

    print()  # New line after countdown
    return '2' if choice[0] is None else choice[0]

def main():
    # Initialize YOLOv8 model
    model = YOLO('yolov8x.pt')
    ct = CentroidTracker(maxDisappeared=40)
    metrics = DetectionMetrics()

    # Set up the WebDriver
    driver = webdriver.Chrome()
    driver.get(f"http://{CCTV_URL}/index.html?Language=0&ViewMode=pull")
    driver.maximize_window()
    time.sleep(20)

    fps = 15

    # Define directories
    base_directory = os.path.join(os.path.dirname(os.path.dirname(__file__)), "screenshots")
    screenshot_directory = os.path.join(base_directory, "Screenshot_sample")
    processed_directory = os.path.join(base_directory, "Screenshot_detection")
    thresholded_directory = os.path.join(base_directory, "Thresholded_images")

    # Create directories
    for directory in [screenshot_directory, processed_directory, thresholded_directory]:
        os.makedirs(directory, exist_ok=True)

    # Get initial screenshot for area setup
    screenshot = driver.get_screenshot_as_png()
    nparr = np.frombuffer(screenshot, np.uint8)
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Initialize polygons list and current polygon
    polygons = []
    current_polygon = []

    # Get user choice with timeout
    print("\nChoose how to define detection areas:")
    print("1: Click manually to create polygons")
    print("2: Load polygons from CSV file")
    print(f"Automatically choosing CSV file after {DEFAULT_TIMEOUT} seconds...")

    choice = get_user_choice(DEFAULT_TIMEOUT)

    if choice == '1':
        # Manual polygon creation
        cv2.imshow("Screenshot", img_bgr)
        param = {
            'image': img_bgr,
            'current_polygon': current_polygon,
            'polygons': polygons
        }
        cv2.setMouseCallback("Screenshot", get_points, param)
        print("Click to draw detection areas. Each polygon needs 4 points. Press 'q' when done.")

        while True:
            key = cv2.waitKey(1)
            if key == ord('q'):
                if len(current_polygon) == 0 and len(polygons) > 0:
                    break
                elif len(current_polygon) > 0:
                    print("Please complete current polygon (need 4 points) or clear it by restarting.")
    else:
        # Load from CSV
        print("Loading polygons from CSV file...")
        polygons = load_coordinates_from_csv()
        if not polygons:
            print("No valid polygons found in CSV. Please run the program again and create polygons manually.")
            driver.quit()
            return

        # Draw loaded polygons
        for polygon in polygons:
            for i in range(len(polygon)):
                cv2.circle(img_bgr, polygon[i], 5, (0, 0, 255), -1)
                if i > 0:
                    cv2.line(img_bgr, polygon[i-1], polygon[i], (0, 255, 0), 2)
            cv2.line(img_bgr, polygon[-1], polygon[0], (0, 255, 0), 2)

        cv2.imshow("Screenshot", img_bgr)
        cv2.waitKey(2000)  # Show loaded polygons for 2 seconds

    cv2.destroyWindow("Screenshot")

    # Create detection area mask from all polygons
    area_mask = create_detection_areas(polygons, img_bgr.shape)

    # Initialize cumulative counters
    total_frames = 0
    screenshot_count = 0

    # Start the main detection loop
    while True:
        screenshot_count += 1
        current_time = time.time()

        metrics.reset()

        screenshot = driver.get_screenshot_as_png()
        nparr = np.frombuffer(screenshot, np.uint8)
        img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        screenshot_path = os.path.join(screenshot_directory, f"screenshot_{screenshot_count}.png")
        cv2.imwrite(screenshot_path, img_bgr)

        cleanup_directory(screenshot_directory, "screenshot_*.png", 5)

        # Draw all detection areas
        for polygon in polygons:
            pts = np.array(polygon, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(img_bgr, [pts], True, (0, 255, 0), 2)

        results = model.predict(source=img_bgr, classes=[0])  # Detect only people

        inputCentroids = []

        for result in results:
            boxes = result.boxes.xyxy
            for box in boxes:
                x1, y1, x2, y2 = map(int, box)
                centroid = (int((x1 + x2) / 2), int((y1 + y2) / 2))
                inputCentroids.append(centroid)

                # Check if centroid is in any detection area
                in_area = check_point_in_area(centroid, area_mask)

                metrics.update(in_area)

                color = (0, 0, 255) if in_area else (255, 0, 0)
                cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)
                cv2.circle(img_bgr, centroid, 5, color, -1)

        total_frames += 1

        objects = ct.update(np.array(inputCentroids))

        cv2.putText(img_bgr, f"Current Frame #{screenshot_count}:", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(img_bgr, f"  Total Detected: {metrics.total_detected}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(img_bgr, f"  In Area: {metrics.crossed_count}", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(img_bgr, f"  Outside Area: {metrics.not_crossed_count}", (10, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        processed_path = os.path.join(processed_directory, f"processed_{screenshot_count}.png")
        cv2.imwrite(processed_path, img_bgr)

        cleanup_directory(processed_directory, "processed_*.png", 5)

        if (current_time - metrics.last_api_call >= 10 or metrics.crossed_since_last_call > 0):
            # Check CCTV connection
            cctv_status = check_cctv_connection()

            # Send data to API if there were crossings or 10 seconds have passed
            if cctv_status:
                if metrics.crossed_since_last_call > 0:
                    create_alarm_api(
                        image_path=processed_path,
                        description=f"Intruder Number:{metrics.crossed_since_last_call}",
                    )

                    # Reset API call timer and crossed counter
                    metrics.last_api_call = current_time
                    metrics.crossed_since_last_call = 0

        # Display the frame
        cv2.imshow("Detection", img_bgr)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        # Control frame rate
        time.sleep(1/fps)

    driver.quit()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()